# OrbitCycle™ — Google Colab notebook

Train and run the OrbitCycle sustainability model end-to-end on Google Colab.

**What this notebook does**
1. Installs dependencies and uploads the model file
2. Smoke-tests the pipeline on synthetic data
3. Loads your real telemetry database (upload or Google Drive)
4. Trains the model and reports held-out metrics
5. Generates the **baseline-vs-ML comparison** Section 5.5 of the Challenge 4 guide explicitly asks for
6. Computes the OrbitCycle Sustainability Score (OSS) per satellite
7. Saves the trained model and exports three plots ready for your deck


## 1. Setup


In [ ]:
# Make sure xgboost is installed (Colab usually has it, but pin to be safe)
!pip install -q xgboost scikit-learn pandas numpy joblib


### Upload `orbitcycle_model.py`

Click **Choose Files** below and pick the `orbitcycle_model.py` you got earlier.


In [ ]:
from google.colab import files
print('Upload orbitcycle_model.py:')
uploaded = files.upload()


In [ ]:
from orbitcycle_model import (
    OrbitCyclePipeline,
    FeatureEngineer,
    OrbitCycleTabularModel,
    TelemetryAnomalyDetector,
    SustainabilityKPIWeights,
    orbitcycle_sustainability_score,
    analytical_orbital_lifetime_years,
    generate_synthetic_telemetry,
    validate_schema,
    REQUIRED_COLUMNS,
    OPTIONAL_BUS_COLUMNS,
    EARTH_RADIUS_KM,
)
print('Import OK')
print('Required columns:', REQUIRED_COLUMNS)


## 2. Smoke test on synthetic data

Verify the pipeline trains and predicts before you wire in real data.


In [ ]:
import numpy as np
import pandas as pd

df_synth = generate_synthetic_telemetry(n_satellites=30, days_per_sat=120, seed=0)
print(f'Synthetic dataset: {len(df_synth):,} rows, {df_synth.norad_id.nunique()} satellites')
df_synth.head()


In [ ]:
# Split by satellite to avoid leakage
sat_ids = df_synth['norad_id'].unique()
rng = np.random.default_rng(42)
rng.shuffle(sat_ids)
cut = int(0.8 * len(sat_ids))
train_synth = df_synth[df_synth.norad_id.isin(sat_ids[:cut])].copy()
test_synth  = df_synth[df_synth.norad_id.isin(sat_ids[cut:])].copy()

pipe_synth = OrbitCyclePipeline().fit(train_synth)
metrics = pipe_synth.tabular.evaluate(pipe_synth.fe.transform(test_synth))
print('Synthetic sanity-check metrics:')
for k, v in metrics.items():
    print(f'  {k:>22s}: {v:.4f}')


## 3. Load your telemetry database

Two options — pick whichever is easier.


### Option A — direct CSV upload

In [ ]:
# Option A: upload your CSV directly
from google.colab import files
print('Upload your telemetry CSV:')
uploaded = files.upload()
csv_name = next(iter(uploaded))
df = pd.read_csv(csv_name, parse_dates=['timestamp'])
print(f'Loaded {len(df):,} rows')
df.head()


### Option B — Google Drive (uncomment to use)

In [ ]:
# Option B: load from Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/path/to/your_telemetry.csv', parse_dates=['timestamp'])
# print(f'Loaded {len(df):,} rows')
# df.head()


### Map your columns to the OrbitCycle schema

If your database uses different column names, rename them in the cell below.


In [ ]:
# Edit this dict to match YOUR column names. Leave empty if names already match.
column_mapping = {
    # 'your_id_col':     'norad_id',
    # 'your_epoch_col':  'timestamp',
    # 'your_a_col':      'semi_major_axis_km',
    # 'your_e_col':      'eccentricity',
    # 'your_i_col':      'inclination_deg',
    # 'your_n_col':      'mean_motion_rev_day',
    # 'your_bstar_col':  'bstar',
}
df = df.rename(columns=column_mapping)

# Verify required columns are present
validate_schema(df)
print('Schema OK')
print('Optional bus columns present:',
      [c for c in OPTIONAL_BUS_COLUMNS if c in df.columns])


### Targets for supervised learning

If your database doesn't include `target_lifetime_years` and `target_deorbit_success`,
run the cell below to bootstrap them from the analytical baseline.

> **For the strongest submission**, replace this with SGP4-propagated truth labels
> before final submission. The bootstrap is fine for getting the pipeline running today.


In [ ]:
if 'target_lifetime_years' not in df.columns:
    df['target_lifetime_years'] = df.apply(
        lambda r: analytical_orbital_lifetime_years(
            r['semi_major_axis_km'] - EARTH_RADIUS_KM, r['bstar']
        ), axis=1
    )
    df['target_deorbit_success'] = (df['target_lifetime_years'] <= 25).astype(int)
    print('Bootstrapped targets from analytical baseline.')
else:
    print('Targets already present in dataframe.')

print(df[['target_lifetime_years', 'target_deorbit_success']].describe())


## 4. Train the pipeline

In [ ]:
# Train/test split by satellite (NOT by row) to avoid leakage
sat_ids = df['norad_id'].unique()
rng = np.random.default_rng(42)
rng.shuffle(sat_ids)
cut = int(0.8 * len(sat_ids))
train_df = df[df.norad_id.isin(sat_ids[:cut])].copy()
test_df  = df[df.norad_id.isin(sat_ids[cut:])].copy()
print(f'Train: {len(train_df):,} rows from {len(sat_ids[:cut])} satellites')
print(f'Test:  {len(test_df):,} rows from {len(sat_ids[cut:])} satellites')

pipe = OrbitCyclePipeline().fit(train_df)
test_feat = pipe.fe.transform(test_df)
metrics = pipe.tabular.evaluate(test_feat)
print('\nHeld-out metrics:')
for k, v in metrics.items():
    print(f'  {k:>22s}: {v:.4f}')


## 5. Feature importance — what the model actually learned

Defends the "AI" label in front of judges.


In [ ]:
import matplotlib.pyplot as plt

imp = pipe.tabular.feature_importance(top_k=12)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(imp['feature'][::-1], imp['importance'][::-1], color='#1f77b4')
ax.set_xlabel('Importance')
ax.set_title('Top features driving lifetime prediction')
plt.tight_layout()
plt.savefig('/content/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Baseline vs ML — the comparison Section 5.5 of Challenge 4 requires

**Put this slide in your deck.** Most teams skip it.


In [ ]:
from sklearn.metrics import mean_absolute_error

# Baseline: physics-only analytical lifetime estimate
test_df_eval = test_df.copy()
test_df_eval['baseline_lifetime'] = test_df_eval.apply(
    lambda r: analytical_orbital_lifetime_years(
        r['semi_major_axis_km'] - EARTH_RADIUS_KM, r['bstar']
    ), axis=1
)

# ML predictions
ml_preds = pipe.predict(test_df)
test_df_eval['ml_lifetime'] = ml_preds['predicted_lifetime_years'].values

baseline_mae = mean_absolute_error(test_df_eval['target_lifetime_years'], test_df_eval['baseline_lifetime'])
ml_mae       = mean_absolute_error(test_df_eval['target_lifetime_years'], test_df_eval['ml_lifetime'])
improvement  = (baseline_mae - ml_mae) / baseline_mae * 100 if baseline_mae > 0 else 0

print(f'Baseline MAE       : {baseline_mae:>10.2f} years')
print(f'OrbitCycle ML MAE  : {ml_mae:>10.2f} years')
print(f'Relative reduction : {improvement:>+10.1f}%')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Analytical baseline', 'OrbitCycle ML'],
              [baseline_mae, ml_mae],
              color=['#888888', '#1f77b4'])
ax.set_ylabel('MAE (years)')
ax.set_title('Lifetime prediction error: baseline vs ML')
for b, v in zip(bars, [baseline_mae, ml_mae]):
    ax.text(b.get_x() + b.get_width()/2, v, f'{v:.1f}',
            ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/baseline_vs_ml.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. OrbitCycle Sustainability Score (OSS) — your headline KPI

In [ ]:
predictions = pipe.predict(df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(predictions['orbitcycle_sustainability_score'], bins=40, color='#2ca02c', alpha=0.85)
median_oss = predictions['orbitcycle_sustainability_score'].median()
ax.axvline(median_oss, color='black', linestyle='--',
           label=f'Median = {median_oss:.1f}')
ax.set_xlabel('OrbitCycle Sustainability Score (0–100)')
ax.set_ylabel('Count')
ax.set_title('OSS distribution across all telemetry samples')
ax.legend()
plt.tight_layout()
plt.savefig('/content/oss_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Worst-10 satellites by latest OSS — operational decision support
latest = predictions.sort_values('timestamp').groupby('norad_id').tail(1)
worst10 = latest.nsmallest(10, 'orbitcycle_sustainability_score')[
    ['norad_id', 'altitude_km', 'predicted_lifetime_years',
     'predicted_deorbit_success_prob', 'anomaly_score',
     'orbitcycle_sustainability_score']
]
print('Worst-10 satellites by latest OSS (priority for operator action):')
worst10.round(2)


## 8. Save outputs and download

Trained model + predictions + plots, all downloaded to your machine for the repo.


In [ ]:
pipe.save('/content/orbitcycle.joblib')
predictions.to_csv('/content/orbitcycle_predictions.csv', index=False)

print('Saved:')
print('  /content/orbitcycle.joblib')
print('  /content/orbitcycle_predictions.csv')
print('  /content/feature_importance.png')
print('  /content/baseline_vs_ml.png')
print('  /content/oss_distribution.png')


In [ ]:
# Trigger downloads to your machine
from google.colab import files
for f in ['orbitcycle.joblib',
          'orbitcycle_predictions.csv',
          'feature_importance.png',
          'baseline_vs_ml.png',
          'oss_distribution.png']:
    files.download(f'/content/{f}')


## 9. For your deck and demo video

**Three plots, mapped to the Phase 1 rubric:**

| File | Slide use | Rubric criterion (weight) |
|---|---|---|
| `baseline_vs_ml.png` | Proof the ML beats the conventional approach | Simulation & KPI Evidence (20%) |
| `feature_importance.png` | Defends what the model learned | Technical Feasibility (30%) |
| `oss_distribution.png` | Visualizes the headline KPI | Mission Sustainability Design (30%) |

**Three lines to say in the demo video:**
- "Baseline MAE: X years. OrbitCycle MAE: Y years. Reduction: Z%."
- "OSS aggregates four sustainability components — lifetime compliance, deorbit probability, health, debris-risk — with weights we tuned for [your reason]."
- "AI tool usage is disclosed in the README per Section 7 of the common Phase 1 guidelines."
